# Session 1: soil pushes, the wall must not slide or tip
**A. Real-world problem.** A garden bed is held up by a low wall. Behind the wall the soil
wants to spread out sideways, like a heavy crowd leaning on a fence. The push grows fast
as the soil gets deeper. Will the wall slide away, or tip over its front edge (the toe)?

**Plan.** (1) the push of the soil, (2) the weight of the wall, (3) two ratios that tell us
whether the wall slides or tips, (4) plot them against wall height.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

SOIL_WEIGHT = 18.0      # unit weight of the soil, kN per cubic metre
PUSH_FACTOR = 1.0 / 3.0  # how much of the soil's weight turns into a sideways push (Session 2 explains)
CONCRETE = 24.0         # unit weight of concrete, kN per cubic metre
FRICTION = 0.5          # friction coefficient between the wall base and the ground

## Step 1: the push of the soil
Near the top there is little soil above, so the push is small; at the bottom it is large.
The push on the wall back looks like a triangle. Its total (kN per metre of wall length) is

    push = 0.5 * PUSH_FACTOR * SOIL_WEIGHT * H * H

and it acts one third of the way up from the bottom (the centre of a triangle).
Check by hand for H = 4 m: 0.5 * (1/3) * 18 * 16 = 48 kN per metre.

In [ ]:
def soil_push(H):
    """Total sideways push of the soil on a wall of height H (m), in kN per metre run."""
    return 0.5 * PUSH_FACTOR * SOIL_WEIGHT * H * H


def push_height(H):
    """Height above the base at which the push acts (m)."""
    return H / 3.0

## Step 2: the weight of the wall
A simple rectangular concrete wall of width B and height H weighs
`CONCRETE * B * H` kN per metre. It acts through the middle of the base, B/2 from the toe.

In [ ]:
def wall_weight(B, H):
    return CONCRETE * B * H

## Step 3: slide or tip?
**Sliding.** Friction under the base can resist at most `FRICTION * weight`. The ratio
`friction / push` says how many times stronger the wall is than the push.
**Tipping.** About the toe, the weight tries to keep the wall down (its turning effect is
weight * B/2) and the push tries to tip it (push * H/3). The ratio of the two turning
effects is the tipping ratio. A ratio below 1 means the wall slides or tips; engineers want
a good margin, at least about 1.5 for sliding and 2 for tipping.

In [ ]:
def slide_ratio(B, H):
    return FRICTION * wall_weight(B, H) / soil_push(H)


def tip_ratio(B, H):
    return wall_weight(B, H) * B / 2.0 / (soil_push(H) * push_height(H))

## Step 4: see it
Left: the picture of the wall, the soil wedge, the push arrow and the weight arrow.
Right: both ratios against wall height for a 2 m wide wall.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    H, B = 4.0, 2.0
    print(f"H = {H} m, B = {B} m: push = {soil_push(H):.1f} kN/m at {push_height(H):.2f} m, weight = {wall_weight(B, H):.1f} kN/m")
    print(f"sliding ratio = {slide_ratio(B, H):.2f}, tipping ratio = {tip_ratio(B, H):.2f}")
    print(f"doubling the height to 8 m: push = {soil_push(8):.0f} kN/m")

    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.fill([0, B, B, 0], [0, 0, H, H], color="#9aa5b1", label="wall (concrete)")
    ax.fill([B, B + 4, B], [0, 0, H], color="#d9b382", alpha=0.7, label="soil wedge")
    ax.fill([B, B + 4, B + 4, B], [H, H, 0, 0], color="#d9b382", alpha=0.25)
    ax.annotate("", xy=(B + 0.05, H / 3), xytext=(B + 1.6, H / 3), arrowprops=dict(arrowstyle="->", color="crimson", lw=2.5))
    ax.text(B + 1.7, H / 3, f"push {soil_push(H):.0f} kN/m", color="crimson", va="center")
    ax.annotate("", xy=(B / 2, 0.05), xytext=(B / 2, 1.4), arrowprops=dict(arrowstyle="->", color="navy", lw=2.5))
    ax.text(B / 2, 1.5, f"weight {wall_weight(B, H):.0f}", color="navy", ha="center")
    ax.set_aspect("equal")
    ax.set_xlabel("distance from the toe (m)")
    ax.set_ylabel("height above the base (m)")
    ax.set_title("Soil pushes, the wall resists")
    ax.legend(loc="upper left")
    fig.tight_layout()
    fig.savefig(OUT / "wall_picture.png", dpi=120)

    Hs = np.linspace(1, 8, 71)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(Hs, [slide_ratio(2.0, h) for h in Hs], label="sliding ratio")
    ax2.plot(Hs, [tip_ratio(2.0, h) for h in Hs], label="tipping ratio")
    ax2.axhline(1.0, color="red", ls="--", label="ratio = 1")
    ax2.set_xlabel("wall height H (m)")
    ax2.set_ylabel("safety ratio (no unit)")
    ax2.set_title("A taller wall of the same width is less safe (B = 2 m)")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "ratio_vs_height.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change `H` and `B` in `lesson.py` to 3 m and 1.2 m and compare the printed ratios with the answer to question B in Section F.
2. **Stretch.** Write `min_width_simple(H)` that returns the smallest B (steps of 0.05 m) giving a sliding ratio of at least 1.5 and a tipping ratio of at least 2, and check H = 3 gives 1.25.
3. **Explain.** Explain, using the formulas for the push and the weight, why a taller wall of the same width tips more easily.

## Assessment
- **A. Conceptual.** Why does doubling the height of the soil quadruple the push on the wall, but only double the weight of a wall of the same width?
- **B. Numerical.** A wall 3 m high and 1.2 m wide holds dry soil (same values as in the lesson). Find the sliding ratio and the tipping ratio. Does it meet the guidance of 1.5 and 2?
- **C. Graph and simulation interpretation.** In the graph for a 2 m wide wall, at what heights do the two ratios reach 1? Which failure comes first as the wall gets taller?
- **D. Design and optimisation challenge.** Design: for a 3 m high wall choose the smallest width, in steps of 0.05 m, that gives a sliding ratio of at least 1.5 and a tipping ratio of at least 2. Which ratio decides?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("worked example: H = 4 m gives a push of 0.5 * (1/3) * 18 * 16 = 48 kN/m",
                   abs(L.soil_push(4.0) - 48.0) < 1e-9))
    checks.append(("the push acts at H/3 = 1.333 m above the base", abs(L.push_height(4.0) - 4.0 / 3.0) < 1e-12))
    checks.append(("wall weight for B = 2 m, H = 4 m is 24 * 2 * 4 = 192 kN/m", abs(L.wall_weight(2.0, 4.0) - 192.0) < 1e-9))
    checks.append(("worked example: sliding ratio 2.00 and tipping ratio 3.00 for B = 2, H = 4",
                   abs(L.slide_ratio(2.0, 4.0) - 2.0) < 1e-9 and abs(L.tip_ratio(2.0, 4.0) - 3.0) < 1e-9))
    checks.append(("doubling the height quadruples the push (192 kN/m at 8 m)",
                   abs(L.soil_push(8.0) / L.soil_push(4.0) - 4.0) < 1e-12 and abs(L.soil_push(8.0) - 192.0) < 1e-9))
    checks.append(("hand calculation: H = 3 m, B = 1.2 m gives sliding 1.6 and tipping 1.92",
                   abs(L.slide_ratio(1.2, 3.0) - 1.6) < 1e-9 and abs(L.tip_ratio(1.2, 3.0) - 1.92) < 1e-9))
    # closed forms for B = 2: slide = 4B/H, tip = 12 B^2 / H^2
    h_slide = 4.0 * 2.0
    h_tip = np.sqrt(12.0 * 4.0)
    checks.append(("for B = 2 m the sliding ratio is 1 at H = 8 m and the tipping ratio is 1 at H = 6.93 m",
                   abs(L.slide_ratio(2.0, h_slide) - 1.0) < 1e-9 and abs(L.tip_ratio(2.0, h_tip) - 1.0) < 1e-9
                   and abs(h_tip - 6.928) < 1e-3))
    checks.append(("design: H = 3 m needs B = 1.25 m for sliding 1.5 and tipping 2 (1.20 m is not enough)",
                   L.slide_ratio(1.25, 3.0) >= 1.5 and L.tip_ratio(1.25, 3.0) >= 2.0 and L.tip_ratio(1.2, 3.0) < 2.0))
    checks.append(("challenge: H = 5 m with B = 2.25 m meets 1.5 and 2, but B = 2.0 m tips (1.92)",
                   L.slide_ratio(2.25, 5.0) >= 1.5 and L.tip_ratio(2.25, 5.0) >= 2.0
                   and abs(L.tip_ratio(2.0, 5.0) - 1.92) < 1e-9))
    checks.append(("1.25 m instead of 1.20 m is about 4 percent more width", abs(1.25 / 1.20 - 1.0 - 0.0417) < 1e-3))
    checks.append(("edge: zero height gives zero push", L.soil_push(0.0) == 0.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
